In [1]:
import pandas as pd
import numpy as np
players = pd.read_csv("/Users/pma/Bootcamp/EDA-futbol/src/data/players.csv")
valuations = pd.read_csv("/Users/pma/Bootcamp/EDA-futbol/src/data/player_valuations.csv")
appearances = pd.read_csv("/Users/pma/Bootcamp/EDA-futbol/src/data/appearances.csv")

In [ ]:
# Fechas

players["date_of_birth"] = pd.to_datetime(players["date_of_birth"])
valuations["date"] = pd.to_datetime(valuations["date"])
appearances["date"] = pd.to_datetime(appearances["date"])
# datetime para pasar de str a datetime64ns(objeto temporal de Pandas)

In [ ]:
def construir_temporada(players, valuations, appearances, inicio, fin, ligas=("ES1", "GB1", "IT1", "L1", "FR1"), min_minutos=900):
    inicio, fin = pd.Timestamp(inicio), pd.Timestamp(fin) # Timestamp representa un punto especifico de tiempo. 

    # Rendimiento por jugador en la temporada
    mask = (appearances["competition_id"].isin(ligas) & (appearances["date"] >= inicio) & (appearances["date"] <= fin))
    rend = (appearances[mask].groupby("player_id")[["minutes_played", "goals", "assists"]].sum().reset_index())

    # Última valoración de la temporada
    val = valuations[(valuations["date"] >= inicio) & (valuations["date"] <= fin)]
    val = val.sort_values("date").groupby("player_id").tail(1) # Solo para que muestre el ultimo dato. 
    val = val[["player_id", "market_value_in_eur"]]

    # Unir, calcular edad y rendimiento
    df = rend.merge(val, on="player_id", how="inner")
    df = df.merge(players[["player_id", "name", "date_of_birth", "position"]], on="player_id", how="left")
    df["edad"] = (fin - df["date_of_birth"]).dt.days / 365.25 # Divido por 365.25 por los anios bisiestos.

    df = df[df["position"].isin(["Attack", "Midfield"]) & (df["minutes_played"] >= min_minutos)].copy()
    df["ga_90"] = (df["goals"] + df["assists"]) / df["minutes_played"] * 90 # Multiplico por 90 que son los minutos de un partido sino daria los "ga_90" por minuto.
    df["temporada"] = f"{inicio.year}/{fin.year}" # Lo uso para decir desde que temporada hasta que temporada quiero que tome. 
    
    return df

In [ ]:
temporadas = [("2023-07-01", "2024-06-30"), ("2024-07-01", "2025-06-30"), ("2025-07-01", "2026-06-30")] # Establezco las temporadas a usar.

dfs = [construir_temporada(players, valuations, appearances, inicio, fin) for inicio, fin in temporadas] # Con un bucle for recorro todas las temporadas elegidas.
df_total = pd.concat(dfs, ignore_index=True) # Concateno los DF.

print(df_total.shape)
print(df_total["temporada"].value_counts())


(2579, 11)
temporada
2025/2026    881
2023/2024    873
2024/2025    825
Name: count, dtype: int64


In [ ]:


print((df_total["edad"] < 23).sum()) # Esto cuenta True como 1 y False como 0 asi que me daria el total de jugadores que cumplen la condicion. 
print(df_total[df_total["edad"] < 23].groupby("temporada").size()) # Agrupo por los jugadores de menos de 23 por temporada. 
# DATO: un mismo jugador puede estar en las 3 temporadas. 

455
temporada
2023/2024    159
2024/2025    145
2025/2026    151
dtype: int64


In [9]:
df_total.to_csv("/Users/pma/Bootcamp/EDA-futbol/src/data/dataset_3_temporadas.csv", index=False) # Guardo el dataset ya filtrado asi no tenga que cargar todo de nuevo.(Por si acaso) 

In [ ]:


df_total["tramo_edad"] = pd.cut(df_total["edad"], bins=[0, 23, 28, 100], labels=["menos de 23", "23 a 27", "28 o más"], right=False) # cut crea una columna nueva "tramo_edad"
# que dice EN CUAL de los 3 grupos cae cada jugador, bins separa por tramos, labels es el nombre de cada tramo y right decide que el limite derecho queda afuera.

print(df_total.groupby("tramo_edad", observed=True).size())
print(df_total.groupby("tramo_edad", observed=True)[["market_value_in_eur", "ga_90"]].median())

tramo_edad
menos de 23     455
23 a 27        1174
28 o más        949
dtype: int64
             market_value_in_eur     ga_90
tramo_edad                                
menos de 23           18000000.0  0.275651
23 a 27               15000000.0  0.323510
28 o más               5000000.0  0.297030


In [ ]:
# Resultado: NO SE CONFIRMA.
# - 455 jugadores-temporada con menos de 23 años (sobre 2.579 en total).
# - Correlación del log del valor con ga_90: 0,49 (moderada, positiva).
# - Correlación del log del valor con la edad: -0,20 (débil, negativa).
# - Dentro de este grupo, el valor se asocia más con el rendimiento que con la edad.
#
# Cuidados: entre 17 y 23 años la edad varía poco, así que tiene poco margen
# para mostrar efecto. Una correlación no prueba causa, y el valor de
# Transfermarkt ya tiene en cuenta cómo juega el futbolista.
# ---------------------------------------------------------------

print(df_total["edad"].isna().sum()) # Sumo los datos que son NaN

jovenes = df_total[df_total["edad"] < 23].copy() # trabajo sobre una copia de la tabla(df_total)
jovenes["log_valor"] = np.log10(jovenes["market_value_in_eur"]) # log_valor es el valor en KK euros y uso log10 para determinar cuantas veces vale mas un jugador que otro(no por KK)
print(jovenes[["log_valor", "edad", "ga_90"]].corr()) # corr sirve para hacer una correlacion, entre las columnas, en un DF que va de -1 a 1 donde 1 es una correlacion perfecta. 

# Resultado: H1 se asocia mas con el rendimiento (ga_90) que con la edad por ende H1 no se estaria cumpliendo. 

1
           log_valor      edad     ga_90
log_valor   1.000000 -0.196024  0.487023
edad       -0.196024  1.000000 -0.042137
ga_90       0.487023 -0.042137  1.000000


In [ ]:
# H2: Pasados los 28 años, el valor cae aunque el rendimiento se mantenga.
#
# Resultado: SE SOSTIENE, con matices.
# - Valor mediano: de 15 M (23 a 27 años) a 5 M (28 o más), una caída de ~67%.
# - ga_90 mediano: de 0,32 a 0,30, una caída de ~8%.
# - Correlación del log del valor con la edad: -0,11 (23 a 27) y -0,48 (28 o más).
# - Correlación del log del valor con ga_90: 0,43 en los de 28 o más.
# - Pasados los 28, la edad pasa a asociarse con el valor tanto como el
#   rendimiento, cosa que no ocurre antes. El rendimiento sigue importando.
#
# Cuidados: el rango de edad es mucho más amplio en el tramo de 28 o más.
# Además, los veteranos con 900 minutos o más son los que siguen siendo buenos,
# así que su rendimiento sale algo inflado.
# ---------------------------------------------------------------

for tramo in ["23 a 27", "28 o más"]: # Para recorrer los dos tramos que me faltan de los jugadores.
    sub = df_total[df_total["tramo_edad"] == tramo].copy() # Utilizo una copia del DF para no utilizar el original.
    sub["log_valor"] = np.log10(sub["market_value_in_eur"]) # Lo mismo que hice antes solo que cambiando los datos de los jugadores.
    print(tramo)
    print(sub[["log_valor", "edad", "ga_90"]].corr()["log_valor"]) # Misma linea que antes para determinar la correlacion. 

23 a 27
log_valor    1.000000
edad        -0.107630
ga_90        0.451838
Name: log_valor, dtype: float64
28 o más
log_valor    1.000000
edad        -0.475971
ga_90        0.425941
Name: log_valor, dtype: float64


In [ ]:
# ---------------------------------------------------------------
# H3 (parte A): El rendimiento ofensivo explica el valor de forma distinta
#               según la posición.
#
# Resultado: SE SOSTIENE, con matices.
# - Attack (1.166 jugadores-temporada): ga_90 = 0,51; edad = -0,46.
# - Midfield (1.413): ga_90 = 0,37; edad = -0,48.
# - En los delanteros el rendimiento pesa algo más que la edad; en los
#   mediocampistas, la edad pesa más que el rendimiento.
#
# Cuidados: ga_90 solo mide goles y asistencias, y subestima el aporte de los
# mediocampistas. No se analiza la liga porque df_total no la incluye.

df_total["log_valor"] = np.log10(df_total["market_value_in_eur"])

for pos in ["Attack", "Midfield"]: # uso el for para que recorra por atacante y mediocampista.
    sub = df_total[df_total["position"] == pos]
    print(pos, len(sub))
    print(sub[["log_valor", "edad", "ga_90"]].corr()["log_valor"]) # imprimo dos cosas, primero la posicion y despues la correlacion que hay entre posicion, valor, edad y ga_90

Attack 1166
log_valor    1.000000
edad        -0.460535
ga_90        0.506170
Name: log_valor, dtype: float64
Midfield 1413
log_valor    1.000000
edad        -0.482341
ga_90        0.373535
Name: log_valor, dtype: float64
